# Modifications :
1. Add Structure output parser. 
2. Build Better routing mechanism.
3. Try improving prompts.
4. Try adding memory to it.

## Importing Packages :

In [1]:
import pandas as pd
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from langchain_community.utilities import GoogleSerperAPIWrapper

## Loading API KEYS :

In [2]:
load_dotenv(".env", override=True)  # or ".env" if that’s your file name/location
if not os.getenv("SERPER_API_KEY"):
    raise RuntimeError("Missing SERPER_API_KEY. Put it in .env as: SERPER_API_KEY=...")

if not os.getenv("GROQ_API_KEY"):
    raise RunTimeError("Missing GROQ_API_KEY, Put it in .env file")

## Data Ingestion and Preprocessing :

In [6]:
## Data 1: reading the Comprehensive Medical Q&A Dataset
df_qa = pd.read_csv("datasets\medical_QnA_Dataset.csv")
## Data has 16407 rows, hence we will sample 500 rows for experimentation
df_qa = df_qa.sample(500, random_state=0).reset_index(drop=True)
print(df_qa.shape)
df_qa.head(10)

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Chaitanya\AppData\Local\Temp\ipykernel_89436\3224530333.py:2: SyntaxWarning: invalid escape sequence '\m'
  df_qa = pd.read_csv("datasets\medical_QnA_Dataset.csv")


(500, 3)


,qtype,Question,Answer
0,frequency,How many people are affected by X-linked chond...,The prevalence of X-linked chondrodysplasia pu...
1,treatment,What are the treatments for Kawasaki disease ?,These resources address the diagnosis or manag...
2,genetic changes,What are the genetic changes related to Ellis-...,Ellis-van Creveld syndrome can be caused by mu...
3,symptoms,What are the symptoms of Renal dysplasia-limb ...,What are the signs and symptoms of Renal dyspl...
4,information,What is (are) Fraser syndrome ?,Fraser syndrome is a rare disorder that affect...
5,frequency,How many people are affected by Pompe disease ?,"Pompe disease affects about 1 in 40,000 people..."
6,information,What is (are) Limb-girdle muscular dystrophy ?,Limb-girdle muscular dystrophy is a group of d...
7,treatment,What are the treatments for Acquired Cystic Ki...,If acquired cystic kidney disease is not causi...
8,information,What is (are) early-onset primary dystonia ?,Early-onset primary dystonia is a condition ch...
9,symptoms,"What are the symptoms of Dystonia 7, torsion ?","What are the signs and symptoms of Dystonia 7,..."


In [7]:
# Preparing the Dataframe for Vector DB by combining the Text
df_qa['combined_text'] = (
    "Question: " + df_qa['Question'].astype(str) + ". " +
    "Answer: " + df_qa['Answer'].astype(str) + ". " +
    "Type: " + df_qa['qtype'].astype(str) + ". "
)
df_qa.head()

,qtype,Question,Answer,combined_text
0,frequency,How many people are affected by X-linked chond...,The prevalence of X-linked chondrodysplasia pu...,Question: How many people are affected by X-li...
1,treatment,What are the treatments for Kawasaki disease ?,These resources address the diagnosis or manag...,Question: What are the treatments for Kawasaki...
2,genetic changes,What are the genetic changes related to Ellis-...,Ellis-van Creveld syndrome can be caused by mu...,Question: What are the genetic changes related...
3,symptoms,What are the symptoms of Renal dysplasia-limb ...,What are the signs and symptoms of Renal dyspl...,Question: What are the symptoms of Renal dyspl...
4,information,What is (are) Fraser syndrome ?,Fraser syndrome is a rare disorder that affect...,Question: What is (are) Fraser syndrome ?. Ans...


In [9]:
## Data 2: reading the Medical Device Manuals Dataset
df_medical_device = pd.read_csv("datasets\medical_device_manuals_dataset.csv")
print(df_medical_device.shape)
## Data has 2694 rows, hence we will sample 500 rows for experimentation
df_medical_device = df_medical_device.sample(500, random_state=0).reset_index(drop=True)
print(df_medical_device.shape)

(2694, 16)
(500, 16)


<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Chaitanya\AppData\Local\Temp\ipykernel_89436\1382810286.py:2: SyntaxWarning: invalid escape sequence '\m'
  df_medical_device = pd.read_csv("datasets\medical_device_manuals_dataset.csv")


In [10]:
# Preparing the Dataframe for Vector DB by combining the Text
df_medical_device['combined_text'] = (
    "Device Name: " + df_medical_device['Device_Name'].astype(str) + ". " +
    "Model: " + df_medical_device['Model_Number'].astype(str) + ". " +
    "Manufacturer: " + df_medical_device['Manufacturer'].astype(str) + ". " +
    "Indications: " + df_medical_device['Indications_for_Use'].astype(str) + ". " +
    "Contraindications: " + df_medical_device['Contraindications'].fillna('None').astype(str)
)
df_medical_device.head()

,Device_Name,Model_Number,Manufacturer,Manual_Version,Publication_Date,Device_Class,Regulatory_Approval_ID,Patient_Population,Indications_for_Use,Contraindications,Sterilization_Method,Number_of_Warnings,Number_of_Cautions,Device_Lifetime_Years,Device_Weight_kg,Max_Operating_Temperature_C,combined_text
0,Electrosurgical Unit,Model 1606,Zimmer Biomet,2023-04-Z,2018-04-28,Class IIb,MDR-847127,All,Used for thermal therapy guidance in oncology ...,Contraindicated in presence of radio frequenci...,Hydrogen Peroxide Plasma,10,15,15.0,6.71,27.0,Device Name: Electrosurgical Unit. Model: Mode...
1,Dialysis Machine,X-6538,3M Healthcare,v2.8,2018-04-03,Class I,MDR-416480,Adult (>65),Used for post-operative chemotherapy managemen...,NaN,Gamma Irradiation,8,11,11.0,NaN,17.0,Device Name: Dialysis Machine. Model: X-6538. ...
2,Ventilator,SON230,Sonova,Version 13,2022-02-24,Class III,BLA670706,Adult (>65),Indicated for real-time temperature assessment...,Do not use during general anesthesia administr...,Pre-Sterilized,18,28,11.0,149.42,25.0,Device Name: Ventilator. Model: SON230. Manufa...
3,Dialysis Machine,Max787,Boston Scientific,v4.7,2016-05-08,Class II,BLA131698,Neonatal,Intended for sterilization guidance during min...,"Not recommended during pregnancy, lactation, o...",NaN,10,15,15.0,20.96,20.0,Device Name: Dialysis Machine. Model: Max787. ...
4,Electrosurgical Unit,Plus691,Abbott,2020-03-Q,2015-01-30,Class III,H127393,Pediatric,Intended for life support evaluation in rehabi...,Contraindicated in patients with severe diabet...,Single-Use Sterile,13,17,8.0,7.75,31.0,Device Name: Electrosurgical Unit. Model: Plus...


## Setting Up ChromaDB :

In [11]:
import chromadb
# Setting up the Chromadb
client = chromadb.PersistentClient(path="./chroma_db_new")

In [12]:
# Collection 1 for medical Q&A Dataset
collection1 = client.get_or_create_collection(name="medical_q_n_a")

# Add data to collection
# here the chroma db will use default embeddings (sentence transformers)
collection1.add(
    documents=df_qa['combined_text'].tolist(),
    metadatas=df_qa.to_dict(orient="records"),
    ids=df_qa.index.astype(str).tolist(),
)

In [13]:
# quick check
query = "What are the treatments for Kawasaki disease ?"
results = collection1.query(query_texts=[query],
    n_results=3
)
print(results)

{'ids': [['1', '393', '60']], 'embeddings': None, 'documents': [["Question: What are the treatments for Kawasaki disease ?. Answer: These resources address the diagnosis or management of Kawasaki disease:  - Cincinnati Children's Hospital Medical Center  - Genetic Testing Registry: Acute febrile mucocutaneous lymph node syndrome  - National Heart, Lung, and Blood Institute: How is Kawasaki Disease Treated?   These resources from MedlinePlus offer information about the diagnosis and management of various health conditions:  - Diagnostic Tests  - Drug Therapy  - Surgery and Rehabilitation  - Genetic Counseling   - Palliative Care. Type: treatment. ", 'Question: What are the treatments for Krabbe Disease ?. Answer: There is no cure for Krabbe disease. Results of a very small clinical trial of children with infantile Krabbe disease found that children who received umbilical cord blood stem cells from unrelated donors prior to symptom onset developed with little neurological impairment. Bon

In [14]:
collection2 = client.get_or_create_collection(name="medical_device_manual")
# Add data to collection
# here the chroma db will use default embeddings (sentence transformers)
collection2.add(
    documents=df_medical_device['combined_text'].tolist(),
    metadatas=df_medical_device.to_dict(orient="records"),
    ids=df_medical_device.index.astype(str).tolist(),
)

In [15]:
# quick check
query = "Which devices are suitable for neonatal patients?"

results = collection2.query(query_texts=[query],
    n_results=5
)
print(results)

{'ids': [['218', '149', '235', '67', '429']], 'embeddings': None, 'documents': [['Device Name: Pacemaker. Model: JOH663. Manufacturer: Johnson & Johnson. Indications: Designed for catheterization support in neonatal intensive care environments.. Contraindications: Avoid use in patients with cardiac devices or cochlear implants. Not suitable for chemical exposure areas.', 'Device Name: Infusion Pump. Model: Model 7800. Manufacturer: Intuitive Surgical. Indications: Designed for joint replacement support in neonatal intensive care environments.. Contraindications: Not recommended for use in geriatric patients or those with hepatic dysfunction.', 'Device Name: Anesthesia Machine. Model: STR623. Manufacturer: Stryker. Indications: Designed for tissue biopsy support in neonatal intensive care environments.. Contraindications: Do not use in combination with sedatives or oxygen therapy modalities.', 'Device Name: Orthopedic Implant. Model: PHI214. Manufacturer: Philips Healthcare. Indications

In [17]:
# Setting Up Serper API

search = GoogleSerperAPIWrapper()
print(search.run("What are different types of trade deals in medical sector?"))

Outsourcing activities like coding, transcriptions and billing collections require managed services agreements (MSAs) and outsourcing agreements. Missing: trade | Show results with:trade. 1. Physician employment contracts · 2. Recruitment agreements · 3. Management service arrangements · 4. Medical directorship arrangements · 5. 9 Most Common Types of Healthcare Contracts · 1. Physician Recruitment Contract · 2. Physician Employment Contract · 3. Medical Director Contract. Class of Trade refers to the distribution channels through which pharmaceutical products are purchased. These can include retail pharmacies, hospitals, long- ... Missing: sector? | Show results with:sector?. Medical Equipment Lease Agreements · Lease duration · Payment terms · Maintenance responsibilities · Conditions for renewal or purchase. The trade modes include cross- border delivery of health services via physical and electronic means, and cross-border movement of consumers, professionals, and ... Missing: type

In [18]:
def get_llm_response(prompt: str) -> str:
    """Function to get response from LLM"""
    client_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    response = client_llm.invoke(
        input=[{"role": "user", "content": prompt}]
    )
    return response.content

In [19]:
## testing the LLM response
prompt = "How to use newtons 3rd LaW ?"
response = get_llm_response(prompt)
print(response)

Newton's 3rd Law, also known as the Law of Action and Reaction, states that for every action, there is an equal and opposite reaction. This law can be applied to various situations in physics and engineering. Here's how to use it:

**Understanding the Law:**

The law states that when an object A exerts a force on object B, object B will exert an equal and opposite force on object A. The forces are equal in magnitude, but opposite in direction.

**Mathematical Representation:**

The law can be mathematically represented as:

F₁ = -F₂

where F₁ is the force exerted by object A on object B, and F₂ is the force exerted by object B on object A.

**Steps to Apply Newton's 3rd Law:**

1. **Identify the objects involved**: Determine the two objects that are interacting with each other.
2. **Determine the forces involved**: Identify the forces that are acting between the two objects.
3. **Label the forces**: Label the forces as action (F₁) and reaction (F₂).
4. **Apply the law**: Use the law to

### NOW WE HAVE EACH RETIEVER SET UP AND IT WILL BE USED IN THE LANGGRAPH FLOW. WHICH WILL ACT AS A NODE AND AND THAT TOOO IN A CONDITIONAL MANNER. WE CAN START BUILDING A AGENTIC PIPELEINE WITH THESE RETIREVERS AND OTHER FUNCTIONALITY IN IT.

## AGENTIC RAG FROM LANGGRAPH :


In [21]:
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, Literal

In [22]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")

structured_model_1 = llm.with_structured_output(DiagnosisSchema)

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")

structured_model_2 = llm.with_structured_output(DiagnosisSchema)

In [ ]:
 ### Respond ONLY in structured format:
"""- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
"""
# Examples in the Prompt

In [ ]:
# === Define workflow node functions ===
def retrieve_context_q_n_a(state):
    """Retrieve top documents from ChromaDB Collection 1 (Medical Q&A Data) based on query."""
    print("---RETRIEVING CONTEXT---")
    query = state["query"] # last user message
    results = collection1.query(query_texts=[query], n_results=3)
    context = "\n".join(results["documents"][0])
    state["context"] = context
    state["source"] = "Medical Q&A Collection"
    print(context)
    # Save context in the state for later nodes
    return state

In [ ]:
# === Define workflow node functions ===
def retrieve_context_medical_device(state):
    """Retrieve top documents from ChromaDB Collection 2 (Medical Device Manuals Data) based on query."""
    print("---RETRIEVING CONTEXT---")
    query = state["query"] # last user message
    results = collection2.query(query_texts=[query], n_results=3)
    context = "\n".join(results["documents"][0])
    state["context"] = context
    state["source"] = "Medical Device Manual"
    print(context)
    # Save context in the state for later nodes
    return state

In [ ]:
def web_search(state):
    """Perform web search using Google Serper API."""
    print("---PERFORMING WEB SEARCH---")
    query = state["query"]
    search_results = search.run(query=query)
    state["context"] = search_results
    state["source"] = "Web Search"
    print(search_results)
    return state

In [ ]:
def router(state: GraphState) -> Literal[
    "Retrieve_QnA", "Retrieve_Device", "Web_Search"
]:
    """Agentic router: decides which retrieval method to use."""
    query = state["query"]
    # A lightweight decision LLM - you can replace this with GPT-4o-mini, etc.
    decision_prompt = f"""
    You are a routing agent. Based on the user query, decide where to look for information.
    Options:
    - Retrieve_QnA: if it's about general medical knowledge, symptoms, or treatment.
    - Retrieve_Device: if it's about medical devices, manuals, or instructions.
    - Web_Search: if it's about recent news, brand names, or external data.
    Query: "{query}"
    Respond ONLY with one of: Retrieve_QnA, Retrieve_Device, Web_Search
    """
    router_decision = get_llm_response(decision_prompt).strip()
    print(f"---ROUTER DECISION: {router_decision}---")
    print(router_decision)
    state["source"] = router_decision
    return state

In [ ]:
# Define the routing function for the conditional edge
def route_decision(state: GraphState) -> str:
    return state["source"]

In [ ]:
def build_prompt(state):
    """Construct the RAG-style prompt."""
    print("---AUGMENT (BUILDING GENERATIVE PROMPT)---")
    query = state["query"]
    context = state["context"]
    prompt = f"""
            Answer the following question using the context below.
            Context:
            {context}
            Question: {query}
            please limit your answer in 50 words.
            """
    
    state["prompt"] = prompt
    print(prompt)
    return state

In [ ]:
def call_llm(state):
    """Call your existing LLM function."""
    print("---GENERATE (CALLING LLM)---")
    prompt = state["prompt"]
    answer = get_llm_response(prompt)
    state["response"] = answer
    return state

In [ ]:
def check_context_relevance(state):
    """Determine whether to retrieved context is relevant or not."""
    print("---CONTEXT RELEVANCE CHECKER---")
    query = state["query"]
    context = state["context"]
    relevance_prompt = f"""
            Check the below context if the context is relevent to the user query or.
            ####
            Context:
            {context}
            ####
            User Query: {query}
            Options:
            - Yes: if the context is relevant.
            - No: if the context is not relevant.
            Please answer with only 'Yes' or 'No'
            """
    relevance_decision_value = get_llm_response(relevance_prompt).strip()
    print(f"---RELEVANCE DECISION: {relevance_decision_value}---")
    state["is_relevant"] = relevance_decision_value
    return state

In [ ]:
# Define the check_context_relevance function for the conditional edge
def relevance_decision(state: GraphState) -> str:
    iteration_count = state.get("iteration_count", 0)
    iteration_count += 1
    state["iteration_count"] = iteration_count
    ## Limiting to max 3 iterations
    if iteration_count >= 3:
        print("---MAX ITERATIONS REACHED, FORCING 'Yes'---")
        state["is_relevant"] = "Yes"
    return state["is_relevant"]

In [ ]:
# === Build the workflow ===
## Define the state structure
class GraphState(TypedDict):
    query: str
    context: str
    prompt: str
    response: str
    source: str  # Which retriever/tool was used
    is_relevant: str
    iteration_count: int

In [ ]:
# Start Creating a Workflow
workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("Router", router)
workflow.add_node("Retrieve_QnA", retrieve_context_q_n_a)
workflow.add_node("Retrieve_Device", retrieve_context_medical_device)
workflow.add_node("Web_Search", web_search)
workflow.add_node("Relevance_Checker", check_context_relevance)
workflow.add_node("Augment", build_prompt)
workflow.add_node("Generate", call_llm)

# Define edges
workflow.add_edge(START, "Router")
workflow.add_conditional_edges(
    "Router",
    route_decision,  # this function decides the path dynamically
    {
        "Retrieve_QnA": "Retrieve_QnA",
        "Retrieve_Device": "Retrieve_Device",
        "Web_Search": "Web_Search",
    }
)
workflow.add_edge("Retrieve_QnA", "Relevance_Checker")
workflow.add_edge("Retrieve_Device", "Relevance_Checker")
workflow.add_edge("Web_Search", "Relevance_Checker")
workflow.add_conditional_edges(
    "Relevance_Checker",
    relevance_decision,  # this function decides the path dynamically
    {
        "Yes": "Augment",
        "No": "Web_Search",
    }
)
workflow.add_edge("Augment", "Generate")
workflow.add_edge("Generate", END)

In [ ]:
# Compile the dynamic RAG agent
agentic_rag = workflow.compile()

In [ ]:
# === Run it ===
from IPython.display import Image, display
display(Image(agentic_rag.get_graph().draw_mermaid_png()))

In [ ]:
from pprint import pprint

input_state = {"query": "What are the treatments for Kawasaki disease?"}

final_state = None
for step in agentic_rag.stream(input_state):
    for node_name, state in step.items():
        print(f"Finished running: {node_name}")
        final_state = state

pprint(final_state["response"])